# 🛡️ ShieldBot: Commercial RAG Assistant for Ceylon Shield Insurance
## Step 1: Smart ingestion (R1)
Load every knowledge-base document, parse its YAML front matter into metadata, and check the metadata before building anything on top of it.

In [1]:
import os
import sys
from datetime import date, datetime
from pathlib import Path

import frontmatter
import pandas as pd
from dotenv import load_dotenv

# Find the project root (the folder that holds pyproject.toml).
# The notebook runs from notebooks/, so plain "data/..." paths would fail.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")

DATA_DIR = ROOT / "data"
KB_DIR = DATA_DIR / "ceylon_shield_knowledge_base"
EVAL_CSV = DATA_DIR / "shieldbot_evaluation_set.csv"
STORAGE_DIR = ROOT / "storage"
RESULTS_DIR = ROOT / "results"
LOGS_DIR = ROOT / "logs"

print("Python        :", sys.version.split()[0])
print("Project root  :", ROOT)
print("KB folder OK  :", KB_DIR.exists())
print("Eval CSV OK   :", EVAL_CSV.exists())
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Python        : 3.14.4
Project root  : d:\shieldbot
KB folder OK  : True
Eval CSV OK   : True
API key loaded: True


In [2]:
def clean_value(value):
    """Convert a front-matter value into a type ChromaDB can store (str, int, float, bool)."""
    if value is None:
        return ""
    if isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, (date, datetime)):
        return value.isoformat()                 # 2026-01-01 -> "2026-01-01"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(v) for v in value)  # ["a", "b"] -> "a, b"
    return str(value)


def first_heading(text: str) -> str:
    """Return the first '# ' heading, used as a fallback title."""
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return ""


def load_documents(kb_dir: Path) -> list[dict]:
    """Load every .md file as {'text': body, 'metadata': {...}}."""
    documents = []
    for path in sorted(kb_dir.rglob("*.md")):
        if path.name.lower() == "readme.md":
            continue  # a folder README is not company content

        post = frontmatter.load(path, encoding="utf-8")
        if not post.metadata:
            print(f"⚠️ No front matter, skipped: {path.name}")
            continue

        metadata = {key: clean_value(val) for key, val in post.metadata.items()}
        metadata.setdefault("title", first_heading(post.content) or path.stem)
        metadata["source_file"] = path.name
        metadata["folder"] = path.parent.name
        metadata["path"] = path.relative_to(kb_dir).as_posix()

        documents.append({"text": post.content.strip(), "metadata": metadata})
    return documents


documents = load_documents(KB_DIR)
print("Documents loaded:", len(documents))

Documents loaded: 26


In [3]:
meta_df = pd.DataFrame([d["metadata"] for d in documents])
meta_df["chars"] = [len(d["text"]) for d in documents]

preferred = ["folder", "source_file", "doc_id", "title", "product", "doc_type",
             "effective_date", "status", "audience", "chars"]
cols = [c for c in preferred if c in meta_df.columns] + \
       [c for c in meta_df.columns if c not in preferred]

print("All front-matter fields found:", sorted(meta_df.columns))
display(meta_df[cols])

All front-matter fields found: ['audience', 'chars', 'confidentiality', 'doc_id', 'doc_type', 'effective_date', 'expiry_date', 'folder', 'path', 'product', 'source_file', 'status', 'superseded_by', 'supersedes', 'title', 'version']


,folder,source_file,doc_id,title,product,doc_type,effective_date,status,audience,chars,version,path,expiry_date,superseded_by,supersedes,confidentiality
0,00_company,branch_directory.md,CS-COM-002,Branch Directory,general,branch_info,2026-06-01,active,public,1397,2026.2,00_company/branch_directory.md,NaN,NaN,NaN,NaN
1,00_company,company_overview.md,CS-COM-001,Company Overview,general,company_info,2026-01-01,active,public,1848,2026.1,00_company/company_overview.md,NaN,NaN,NaN,NaN
2,00_company,contact_and_service_hours.md,CS-COM-003,Contact Channels and Service Hours,general,contact_info,2026-01-01,active,public,1029,2026.1,00_company/contact_and_service_hours.md,NaN,NaN,NaN,NaN
3,01_motor,motor_premium_table_2026.md,CS-MOT-003,Motor Premium Rates 2026,motor,premium_table,2026-01-01,active,public,1421,2026.1,01_motor/motor_premium_table_2026.md,NaN,NaN,NaN,NaN
4,01_motor,motorshield_comprehensive_policy_2026.md,CS-MOT-001,MotorShield Comprehensive — Policy Terms,motor,policy_terms,2026-01-01,active,public,3251,2026.1,01_motor/motorshield_comprehensive_policy_2026.md,NaN,NaN,NaN,NaN
5,01_motor,motorshield_third_party_policy_2026.md,CS-MOT-002,MotorShield Third Party — Policy Terms,motor,policy_terms,2026-01-01,active,public,1098,2026.1,01_motor/motorshield_third_party_policy_2026.md,NaN,NaN,NaN,NaN
6,02_health,health_premium_table_2026.md,CS-HLT-003,Health Shield Plus Premium Table 2026,health,premium_table,2026-01-01,active,public,787,2026.1,02_health/health_premium_table_2026.md,NaN,NaN,NaN,NaN
7,02_health,health_shield_plus_policy_2025.md,CS-HLT-001,Health Shield Plus — Policy Terms (2025),health,policy_terms,2025-01-01,superseded,public,1239,2025.1,02_health/health_shield_plus_policy_2025.md,2025-12-31,CS-HLT-002,NaN,NaN
8,02_health,health_shield_plus_policy_2026.md,CS-HLT-002,Health Shield Plus — Policy Terms (2026),health,policy_terms,2026-01-01,active,public,2607,2026.1,02_health/health_shield_plus_policy_2026.md,NaN,NaN,CS-HLT-001,NaN
9,03_life,life_premium_table.md,CS-LIF-002,Life Secure Premium Table,life,premium_table,2026-01-01,active,public,1026,2026.1,03_life/life_premium_table.md,NaN,NaN,NaN,NaN


In [4]:
print("By status:\n", meta_df["status"].value_counts(), "\n")
print("By audience:\n", meta_df["audience"].value_counts(), "\n")
print("By product:\n", meta_df["product"].value_counts(), "\n")

print("Superseded docs:", meta_df.loc[meta_df["status"] == "superseded", "source_file"].tolist())
print("Internal docs  :", meta_df.loc[meta_df["audience"] == "internal", "source_file"].tolist())

# Any document missing a field the filters depend on?
REQUIRED = ["doc_id", "title", "product", "doc_type", "effective_date", "status", "audience"]
missing = {
    d["metadata"]["source_file"]: [f for f in REQUIRED if not d["metadata"].get(f)]
    for d in documents
}
missing = {k: v for k, v in missing.items() if v}
print("Docs missing required fields:", missing or "none ✅")

# Done-when checks
assert len(documents) == 26, f"Expected 26 docs, got {len(documents)}"
assert (meta_df["status"] == "superseded").sum() == 1
assert (meta_df["audience"] == "internal").sum() == 2
print("\n✅ Step 1 checks passed")

By status:
 status
active        25
superseded     1
Name: count, dtype: int64 

By audience:
 audience
public      24
internal     2
Name: count, dtype: int64 

By product:
 product
general    12
motor       4
health      4
life        3
travel      2
home        1
Name: count, dtype: int64 

Superseded docs: ['health_shield_plus_policy_2025.md']
Internal docs  : ['agent_handbook.md', 'fraud_awareness_guidelines.md']
Docs missing required fields: none ✅

✅ Step 1 checks passed
